# Testing functions

## Load the GTFS data

In [1]:
import pandas as pd
import zipfile


def load_data(file_path):
    """From a ZIP file, unzip and load the CSV files (with .txt extension) into Pandas DataFrames, following a specific GTFS naming convention: agency, stops, routes, trips, shapes, etc."""
    tables = {}
    with zipfile.ZipFile(file_path, "r") as z:
        for filename in z.namelist():
            if filename.endswith(".txt") and not filename.startswith("__MACOS"):
                with z.open(filename) as f:
                    df_name = filename.split(".")[0]
                    tables[df_name] = pd.read_csv(f)
    return tables


# Load the data from the ZIP file in ./assets/GTFS_bUCR.zip
gtfs_data = load_data("./assets/GTFS_bUCR.zip")

## Show a glimpse of the data

In [2]:
gtfs_data["shapes"].head()  # Display the first few rows of the routes DataFrame


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,desde_educacion_sin_milla,9.935549,-84.049114,0,0.000
1,desde_educacion_sin_milla,9.935559,-84.049158,1,0.005
2,desde_educacion_sin_milla,9.935574,-84.049224,2,0.012
3,desde_educacion_sin_milla,9.935600,-84.049325,3,0.024
4,desde_educacion_sin_milla,9.935638,-84.049417,4,0.035


## Convert to GeoDataFrames

Convert the `shapes` DataFrame to a GeoDataFrame with LineString geometries.

In [3]:
import geopandas as gpd

shapes = gtfs_data["shapes"]
shapes = gpd.GeoDataFrame(
    shapes, geometry=gpd.points_from_xy(shapes["shape_pt_lon"], shapes["shape_pt_lat"])
)
shapes = shapes.dissolve(by="shape_id", as_index=False)

# Convert the geometry from Multipoint to LineString
shapes["geometry"] = shapes["geometry"].apply(lambda x: x.convex_hull)

In [4]:
shapes

,shape_id,geometry,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,desde_artes_con_milla,"POLYGON ((-84.04561 9.93462, -84.05222 9.93551...",9.935511,-84.052220,0,0.0
1,desde_artes_sin_milla,"POLYGON ((-84.04561 9.93461, -84.05146 9.93528...",9.935512,-84.052232,0,0.0
2,desde_educacion_con_milla,"POLYGON ((-84.04562 9.93462, -84.05219 9.9355,...",9.935546,-84.049111,0,0.0
3,desde_educacion_sin_milla,"POLYGON ((-84.04561 9.93462, -84.05146 9.93528...",9.935549,-84.049114,0,0.0
4,hacia_artes,"POLYGON ((-84.04562 9.93465, -84.05198 9.93547...",9.946514,-84.045279,0,0.0
5,hacia_educacion,"POLYGON ((-84.04562 9.93465, -84.04711 9.93496...",9.946515,-84.045271,0,0.0


## Show a map with the geometries in the GeoDataFrame

In [5]:
# Show a map with the geometries in the GeoDataFrame
shapes.explore()
